In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
from statsmodels.tools import add_constant
from statsmodels.regression.linear_model import OLS

plt.rcParams.update({'font.size': 20})

TICKER_A    = 'V'
TICKER_B    = 'MA'
PAIR_NAME   = 'V_MA'
TRAIN_START = '2009-01-01'
TRAIN_END   = '2022-12-31'

raw = yf.download(
    [TICKER_A, TICKER_B],
    start       = TRAIN_START,
    end         = TRAIN_END,
    auto_adjust = True,
    progress    = False
)['Close'].dropna()

log_A = np.log(raw[TICKER_A].values)
log_B = np.log(raw[TICKER_B].values)
dates = raw.index

res        = OLS(log_A, add_constant(log_B)).fit()
alpha_hat  = float(res.params[0])
beta_hat   = float(res.params[1])
print(f'OLS - beta={beta_hat:.4f}  alpha={alpha_hat:.5f}')

counterpart = beta_hat * log_B + alpha_hat

fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(dates, log_A,       color='steelblue', lw=0.9,
        label=f'$\\log p^A$ ({TICKER_A})')
ax.plot(dates, counterpart, color='tomato',    lw=0.9, ls='--',
        label=f'$\\hat{{\\beta}}\\,\\log p^B + \\hat{{\\alpha}}$'
              f' ({TICKER_B})')

ax.set_ylabel('Log price', fontsize=20)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.tick_params(axis='both', labelsize=11)
ax.legend(fontsize=20)
ax.grid(True, alpha=0.3)
plt.tight_layout()

import os
os.makedirs('figures', exist_ok=True)
plt.savefig(f'figures/{PAIR_NAME}_logprices.png', dpi=150,
            bbox_inches='tight')
plt.show()